# Module 2 — Analytics: Exploratory Data Analysis (Titanic Dataset)

This notebook performs end-to-end data profiling, threshold-based missing value handling, univariate distribution analysis, bivariate survival rates using boolean masks, the exact 6-column correlation heatmap, multivariate data storytelling, and an EDA standardization sanity check.

**Critical Data Ingestion Rule**: The dataset is loaded strictly from the offline local fallback `titanic.csv` (which was generated once via `sns.load_dataset('titanic')`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
CSV_PATH = Path("titanic.csv")

# Load dataset strictly from offline CSV
df = pd.read_csv(CSV_PATH)
print("Loaded Titanic shape:", df.shape)
df.head()


## 1. Profiling & Missing-Value Analysis
We inspect schema information, numerical distributions, and compute exact missing-value percentages across all columns.


In [ ]:
print("DataFrame Info:")
df.info()

print("\nSummary Statistics:")
df.describe()


In [ ]:
# Missing Value Percentages
missing_counts = df.isnull().sum()
missing_pcts = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({
    "Missing_Count": missing_counts[missing_counts > 0],
    "Missing_Percentage": missing_pcts[missing_pcts > 0].round(2)
})
missing_df


### Threshold-Based Missing Value Strategy Justification:
- **< 5% Missing (`embarked`, `embark_town` at 0.22%)**: Only 2 records lack embarkation port. For EDA we drop these 2 rows; for modeling pipelines we impute the most frequent port ('S').
- **5% – 30% Missing (`age` at 19.87%)**: 177 passengers have missing age. Because the age distribution exhibits positive skew, median imputation (28.0 years) is optimal over mean imputation to avoid distortion.
- **> 30% Missing (`deck` at 77.22%)**: Over 77% missing data prevents reliable inference without introducing severe synthetic bias. We drop the `deck` column.


## 2. Univariate Analysis: Age & Fare
We analyze distributions, compute IQR outlier boundaries, and examine Fare central tendencies.


In [ ]:
# Age Univariate Analysis & IQR Outliers
age_valid = df["age"].dropna()
q1_age = age_valid.quantile(0.25)
q3_age = age_valid.quantile(0.75)
iqr_age = q3_age - q1_age
lower_age = q1_age - 1.5 * iqr_age
upper_age = q3_age + 1.5 * iqr_age
outliers_age = age_valid[(age_valid < lower_age) | (age_valid > upper_age)]

print(f"Age IQR: Q1={q1_age:.2f}, Q3={q3_age:.2f}, IQR={iqr_age:.2f}")
print(f"Age Outlier Thresholds: [{lower_age:.2f}, {upper_age:.2f}]")
print(f"Age Outliers Count: {len(outliers_age)} ({len(outliers_age)/len(age_valid)*100:.2f}%)")


In [ ]:
# Fare Univariate Analysis, IQR Outliers & Central Tendencies
fare_valid = df["fare"].dropna()
q1_fare = fare_valid.quantile(0.25)
q3_fare = fare_valid.quantile(0.75)
iqr_fare = q3_fare - q1_fare
lower_fare = q1_fare - 1.5 * iqr_fare
upper_fare = q3_fare + 1.5 * iqr_fare
outliers_fare = fare_valid[(fare_valid < lower_fare) | (fare_valid > upper_fare)]

fare_mean = fare_valid.mean()
fare_median = fare_valid.median()
fare_mode = fare_valid.mode()[0]

print(f"Fare Mean: {fare_mean:.2f}")
print(f"Fare Median: {fare_median:.2f}")
print(f"Fare Mode: {fare_mode:.2f}")
print(f"Fare Outliers Count: {len(outliers_fare)} ({len(outliers_fare)/len(fare_valid)*100:.2f}%)")


### Skewness Conclusion:
Since **Mean (32.20) > Median (14.45) > Mode (8.05)**, the `fare` distribution exhibits substantial **positive (rightward) skewness**. The long right tail is created by expensive First-Class luxury suites costing up to £512.33, while the vast majority of third-class tickets clustered around £7.91 – £8.05.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age plots
sns.histplot(age_valid, kde=True, ax=axes[0, 0], color="skyblue")
axes[0, 0].set_title(f"Age Distribution (Outliers: {len(outliers_age)})")
sns.boxplot(x=age_valid, ax=axes[0, 1], color="lightblue")
axes[0, 1].set_title("Age Boxplot")

# Fare plots
sns.histplot(fare_valid, kde=True, ax=axes[1, 0], color="coral", bins=30)
axes[1, 0].axvline(fare_mean, color="red", linestyle="--", label=f"Mean ({fare_mean:.1f})")
axes[1, 0].axvline(fare_median, color="green", linestyle="-", label=f"Median ({fare_median:.1f})")
axes[1, 0].axvline(fare_mode, color="blue", linestyle=":", label=f"Mode ({fare_mode:.1f})")
axes[1, 0].set_title("Fare Distribution (Right-Skewed)")
axes[1, 0].legend()
sns.boxplot(x=fare_valid, ax=axes[1, 1], color="salmon")
axes[1, 1].set_title(f"Fare Boxplot (Outliers: {len(outliers_fare)})")

plt.tight_layout()
plt.show()


## 3. Bivariate Analysis: Survival Rates via Boolean Masking
We compute survival rates by gender, passenger class, and gender + passenger class using boolean conditions.


In [ ]:
# 1. Survival by Sex
sr_female = df[df["sex"] == "female"]["survived"].mean()
sr_male = df[df["sex"] == "male"]["survived"].mean()

# 2. Survival by Pclass
sr_p1 = df[df["pclass"] == 1]["survived"].mean()
sr_p2 = df[df["pclass"] == 2]["survived"].mean()
sr_p3 = df[df["pclass"] == 3]["survived"].mean()

# 3. Survival by Sex AND Pclass
sr_f_p1 = df[(df["sex"] == "female") & (df["pclass"] == 1)]["survived"].mean()
sr_f_p2 = df[(df["sex"] == "female") & (df["pclass"] == 2)]["survived"].mean()
sr_f_p3 = df[(df["sex"] == "female") & (df["pclass"] == 3)]["survived"].mean()
sr_m_p1 = df[(df["sex"] == "male") & (df["pclass"] == 1)]["survived"].mean()
sr_m_p2 = df[(df["sex"] == "male") & (df["pclass"] == 2)]["survived"].mean()
sr_m_p3 = df[(df["sex"] == "male") & (df["pclass"] == 3)]["survived"].mean()

print(f"Survival Rate by Sex:\n  Female: {sr_female*100:.2f}% | Male: {sr_male*100:.2f}%\n")
print(f"Survival Rate by Pclass:\n  Class 1: {sr_p1*100:.2f}% | Class 2: {sr_p2*100:.2f}% | Class 3: {sr_p3*100:.2f}%\n")
print("Survival Rate by Sex AND Pclass:")
print(f"  Female Class 1: {sr_f_p1*100:.2f}% | Male Class 1: {sr_m_p1*100:.2f}%")
print(f"  Female Class 2: {sr_f_p2*100:.2f}% | Male Class 2: {sr_m_p2*100:.2f}%")
print(f"  Female Class 3: {sr_f_p3*100:.2f}% | Male Class 3: {sr_m_p3*100:.2f}%")


## 4. Correlation Matrix: EXACT 6 Columns
As required, the correlation matrix and heatmap strictly contain:
`survived`, `pclass`, `age`, `sibsp`, `parch`, `fare` (explicitly excluding `adult_male` and `alone`).


In [ ]:
corr_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, square=True)
plt.title("6x6 Correlation Heatmap (Excluding adult_male and alone)")
plt.show()

corr_matrix.round(4)


### Interpretation of Two Strongest Off-Diagonal Correlations:
1. **`pclass` and `fare` ($r = -0.5495$, $|r| = 0.5495$)**:
   The strongest correlation in the dataset is negative. As passenger class number increases ($1 	o 2 	o 3$, indicating lower travel class), the ticket fare drops steeply. First-class berths required substantially higher financial investment than steerage cabins.
2. **`sibsp` and `parch` ($r = +0.4148$, $|r| = 0.4148$)**:
   The second strongest correlation is positive. Passengers accompanied by siblings or spouses were significantly more likely to also travel with parents or children, reflecting intact family units traveling together.


## 5. Multivariate Data Story (4 Distinct Charts & Interpretations)


In [ ]:
# Chart 1: Survival by Class and Gender
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="pclass", y="survived", hue="sex", ci=None, palette=["#e74c3c", "#3498db"])
plt.title("Chart 1: Survival Probability by Class and Gender")
plt.ylabel("Survival Rate")
plt.show()


**Chart 1 Interpretation:**
Female passengers had dramatically higher survival rates across all three passenger classes compared to males, reflecting the maritime evacuation protocol of prioritizing women and children. However, socioeconomic class remained a strong modifier: First-class women survived at a staggering 96.8% rate, while third-class women survived at only 50.0%. For men, survival dropped from 36.9% in first class down to 13.5% in third class, underscoring the compounding advantage of cabin placement and deck access during evacuation.


In [ ]:
# Chart 2: Fare Distribution across Class and Survival
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="pclass", y="fare", hue="survived", palette="Set2", showfliers=False)
plt.title("Chart 2: Ticket Fare Distribution by Class and Survival Status")
plt.ylabel("Fare (£)")
plt.show()


**Chart 2 Interpretation:**
Within each passenger class, individuals who survived consistently paid higher median fares than those who perished. This was most pronounced in first class, where survivors occupied premium mid-deck and upper-deck staterooms that provided faster and less obstructed physical access to the boat deck. In second and third class, higher fares often correlated with larger family berths or slightly elevated deck accommodations that improved evacuation odds.


In [ ]:
# Chart 3: Age vs Fare Scatter segmented by Survival
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="age", y="fare", hue="survived", style="sex", alpha=0.8, palette={0: "#c0392b", 1: "#27ae60"})
plt.title("Chart 3: Passenger Age vs. Fare Segmented by Survival Status & Gender")
plt.xlabel("Age (years)")
plt.ylabel("Fare (£)")
plt.show()


**Chart 3 Interpretation:**
The scatter visualization confirms that high-fare passengers (primarily located in the upper portion of the plot) survived at remarkably high proportions, especially female passengers regardless of their age group. Young children (under 10 years old) achieved noticeable survival clusters in second and third classes, confirming preferential lifeboat access. Conversely, adult males between ages 20 and 50 paying lower fares (clustered tightly near the bottom axis) experienced the highest fatality density.


In [ ]:
# Chart 4: Embarkation Port and Class Survival Distribution
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="embarked", y="survived", hue="pclass", ci=None, palette="Blues")
plt.title("Chart 4: Survival Probability by Embarkation Port and Class")
plt.ylabel("Survival Rate")
plt.show()


**Chart 4 Interpretation:**
Passengers embarking at Cherbourg ('C') enjoyed the highest aggregate survival rates because a large plurality boarded directly into first-class accommodations. In contrast, Southampton ('S') and Queenstown ('Q') saw heavy representation of third-class emigrant passengers who were housed in lower decks with labyrinthine corridor access to lifeboats, resulting in significantly lower survival percentages.


## 6. EDA Standardization Sanity Check
We compute z-scores for `age` and `fare` strictly as an exploratory confirmation of standard normal centering.

$$\mu \approx 0, \quad \sigma \approx 1$$

*Crucial Architecture Note*: This transformation is purely an exploratory sanity check. It is completely isolated and NOT fed into modeling to avoid data leakage.


In [ ]:
age_mean, age_std = age_valid.mean(), age_valid.std()
fare_mean_s, fare_std_s = fare_valid.mean(), fare_valid.std()

z_age = (age_valid - age_mean) / age_std
z_fare = (fare_valid - fare_mean_s) / fare_std_s

print(f"Age Before: Mean = {age_mean:.4f}, Std = {age_std:.4f}")
print(f"Age Z-Score: Mean = {z_age.mean():.4f} (~0), Std = {z_age.std():.4f} (~1)\n")
print(f"Fare Before: Mean = {fare_mean_s:.4f}, Std = {fare_std_s:.4f}")
print(f"Fare Z-Score: Mean = {z_fare.mean():.4f} (~0), Std = {z_fare.std():.4f} (~1)")
